# Governance rule back-tests (rules A-1, H-1, H-2, B-1)

Based on the INFO 4340 Class 18 threshold back-testing notebook. A rule only counts as governance once you know:

1. **How often would it fire?** If it flags almost nothing, it does nothing. If it flags half the volume, nobody can keep up.
2. **Does it fire on the errors you care about?** A rule can flag a reasonable share and still miss most mistakes.

Each rule from the governance memo is tested on the project's real outputs: the Claude batch run (`data/llm/batch_v2_analysis.csv`, 5,667 reports), the 100-item human eval set (`data/labels/labels_partner*.xlsx` + `data/llm/eval_v2.csv`), and the cleaned data. Nothing here calls the API.

| Rule | Domain | Kind of rule | What "fire" means here |
|---|---|---|---|
| A-1 Contamination check | Accuracy & evaluation | Run-level, with an item-level part (recognized / leaked items) | Any of T1–T3 crosses its threshold |
| H-1 Human field review | Human review | Field-level, on the audit sample | A field's κ is below 0.60 |
| H-2 Human output review | Human review | Row-level | More than 5% of rows are flagged |
| B-1 School-status disparity | Bias & fairness | Group-level, within each real draft tier | Disparity ratio > 1.25, or Claude's ratio exceeds the grade's ratio by 0.10 or more |

**Two meanings of "error"** (used for the "errors caught" questions):
- **Audit error (eval set, n = 100):** Claude's `projected_tier` differs from **both** partners. This is the best measure of a labeling mistake, but it exists only for the audit sample.
- **Outcome miss (full batch, n = 5,667):** Claude's `projected_tier` differs from the player's real draft tier. This is a proxy: Claude labels what the *text* projects, and the text can be wrong about the outcome. It is the only error measure available for every row.

## Setup

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# The notebook lives in src/; find the repo root so `src` imports and data paths work from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "cleaning.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import cleaning as C
from src import evaluation as E
from src import final_comparison as FC
from src import labeling as L
from src import modeling as M

TIERS = list(C.TIERS)                       # ["R1", "R2-3", "R4-7", "UDFA"]
TIER_CODE = {t: i for i, t in enumerate(TIERS)}
ROLE_CODE = L.ORDINAL["role"]
SEED = 4340
pd.set_option("display.width", 160)

# ---- thresholds as written in the governance memo (change here to re-test)
A1_GAP, A1_MIN_RECOGNIZED = 0.05, 30        # T1: recognized-minus-unrecognized accuracy gap
A1_MDD = 0.029                              # T2: max allowed change in any kappa / Model E macro-F1
A1_RECOG_RATE = 0.10                        # T3: recognition rate in any class year
H1_KAPPA = 0.60                             # H-1: field kappa floor
H1_AUDIT_MIN, H1_AUDIT_SHARE = 100, 0.02    # H-1: audit size = max(100, 2% of the run)
H2_ROW_RATE, H2_TIER_GAP = 0.05, 2          # H-2: share of flagged rows; tier gap vs grade-implied tier
B1_RATIO, B1_EXCESS, B1_MIN_N = 1.25, 0.10, 30

# Eval items with unmasked identifying text, flagged by the human labelers (docs/masking_limitations.md)
LEAK_ITEMS = ["E011", "E031", "E042", "E069", "E085", "E094"]

SUMMARY = []   # one row per trigger, filled in as each rule is back-tested


def record(rule, trigger, value, threshold, fired, note=""):
    SUMMARY.append({"rule": rule, "trigger": trigger, "value": value, "threshold": threshold,
                    "fired": bool(fired), "note": note})


def show_backtest(name, flag, is_error):
    # Class 18 back-test: flag rate and share of errors caught
    r = E.backtest_flag(pd.Series(flag).reset_index(drop=True), pd.Series(is_error).reset_index(drop=True))
    print(f"{name}")
    print(f"  Flags:   {r['flagged']}/{len(flag)} items ({r['flag_rate']:.1%})")
    print(f"  Catches: {r['caught']}/{r['errors']} errors ({r['catch_rate']:.0%})"
          f"   precision of flag: {r['precision_of_flag']:.2f}" if r["flagged"] else
          f"  Catches: 0/{r['errors']} errors")
    return r

## Load the project results

- `b`: the full Claude batch (one row per analysis report) joined to the real tier, grade, school status and the masked text Claude saw.
- `ev`: the 100 audit items, with Claude's eval-run labels and both partners' labels. The partners' workbooks contain only masked text (no names), so items are matched to players by their masked report text.
- **Grade-implied tier** (used by H-2 and B-1): rank players by `grade_pct` (the analyst's grade as a percentile within their class year), then cut into tiers using the tier shares of the 2015–22 training years. The top 7–8% of each class is "R1 by grade", and so on.

In [8]:
clean = pd.read_parquet(C.ROOT / "data" / "prospects_clean.parquet")
clean["tier"] = clean["tier"].astype(str)

train = clean[(clean.split == "train") & clean.in_analysis.astype(bool)]
cum_share = np.cumsum(train.tier.value_counts(normalize=True).reindex(TIERS).values)


def grade_tier(grade_pct):
    # highest grade_pct -> R1; cut points are the training-year tier shares
    return np.searchsorted(cum_share, 1 - np.asarray(grade_pct, dtype=float)).clip(0, 3)


def add_context(df):
    cols = C.KEY + ["tier", "split", "grade_pct", "power5", "positionGroup", "scout_text", "collegeNames"]
    out = df.merge(clean[cols], on=C.KEY, how="left", validate="one_to_one")
    out["real"] = out.tier.map(TIER_CODE)
    out["cl"] = out.projected_tier.map(TIER_CODE)
    out["grade_t"] = grade_tier(out.grade_pct)
    out["text_masked"] = [C.mask_text(t, n, c) for t, n, c in zip(out.scout_text, out.displayName, out.collegeNames)]
    out["recognized"] = out.recognized.astype(str).str.lower().eq("true")
    return out


b = add_context(pd.read_csv(C.ROOT / "data" / "llm" / "batch_v2_analysis.csv"))
b["outcome_miss"] = b.cl != b.real

# ---- audit sample: Claude eval run + both partners
ws = lambda s: " ".join(str(s).split())
ev = add_context(pd.read_csv(C.ROOT / "data" / "llm" / "eval_v2.csv"))
ev["_k"] = ev.text_masked.map(ws)
for p in ["partner1", "partner2"]:
    lab = L.read_labels(p)
    lab = lab[lab.set == "eval"].assign(_k=lambda x: x.report.map(ws))
    keep = ["_k", "projected_tier", "role", "injury_flag", "character_flag", "tone"] + (["item_id"] if p == "partner1" else [])
    ev = ev.merge(lab[keep].rename(columns={f: f"{f}_{p[-1]}" for f in keep[1:6]}), on="_k", how="left", validate="one_to_one")
assert ev.projected_tier_1.notna().all() and ev.projected_tier_2.notna().all(), "audit items not matched to partner labels"
ev["h1"], ev["h2"] = ev.projected_tier_1.map(TIER_CODE), ev.projected_tier_2.map(TIER_CODE)
ev["audit_error"] = (ev.cl != ev.h1) & (ev.cl != ev.h2)
ev["leaked"] = ev.item_id.isin(LEAK_ITEMS)

print(f"Batch rows: {len(b):,}   outcome misses: {b.outcome_miss.sum():,} ({b.outcome_miss.mean():.1%})")
print(f"Audit items: {len(ev)} matched to both partners   audit errors (Claude differs from both): "
      f"{ev.audit_error.sum()}   partners agree with each other on {(ev.h1 == ev.h2).sum()}")
print(f"Audit items by real tier: {ev.tier.value_counts().reindex(TIERS).to_dict()}")

Batch rows: 5,667   outcome misses: 2,677 (47.2%)
Audit items: 100 matched to both partners   audit errors (Claude differs from both): 29   partners agree with each other on 83
Audit items by real tier: {'R1': 25, 'R2-3': 25, 'R4-7': 25, 'UDFA': 25}


In [9]:
def field_codes(df, field, who):
    # numeric codes for one field; who = "cl", "1" or "2"
    col = field if who == "cl" else f"{field}_{who}"
    s = df[col]
    if field == "projected_tier":
        return s.map(TIER_CODE)
    if field == "role":
        return s.map(ROLE_CODE)
    return pd.to_numeric(s, errors="coerce")


def kappa(df, field, a, b_):
    x, y = field_codes(df, field, a), field_codes(df, field, b_)
    ok = x.notna() & y.notna()
    weights = "quadratic" if field in ("projected_tier", "role", "tone") else None
    return cohen_kappa_score(x[ok].astype(int), y[ok].astype(int), weights=weights)


FIELDS = ["projected_tier", "role", "tone", "injury_flag", "character_flag"]
PAIRS = {"P1 vs P2": ("1", "2"), "Claude vs P1": ("cl", "1"), "Claude vs P2": ("cl", "2")}


def kappa_table(df):
    return pd.DataFrame({pair: {f: kappa(df, f, *who) for f in FIELDS} for pair, who in PAIRS.items()})


K_ALL = kappa_table(ev)
K_ALL.round(3)

,P1 vs P2,Claude vs P1,Claude vs P2
projected_tier,0.907,0.735,0.795
role,0.883,0.758,0.834
tone,0.819,0.711,0.711
injury_flag,0.878,0.878,1.000
character_flag,0.918,0.000,0.000


## Rule A-1: Contamination check (accuracy & evaluation)

Fires if **any** of these is true:
- **T1:** Claude's tier accuracy for players it marks as recognized minus its accuracy for players it doesn't is **> 0.05**, **and** the 95% bootstrap CI lower bound is **> 0** (2024–26 test split, at least 30 recognized players).
- **T2:** Any κ (human-human or human-LLM), or Model E's macro-F1, changes by **more than the MDD of 0.029** when recognized or leaked items are removed.
- **T3:** The recognition rate (recognized ÷ players) in **any class year** is **> 10%**.

**Action:** re-score without recognized and leak-flagged items and report both versions. Treat Claude's accuracy against real outcomes as an upper bound.

### A-1 T1: accuracy gap, recognized vs not recognized

In [10]:
def recog_gap(df, n_boot=2000, seed=SEED):
    hit = (df.cl == df.real).to_numpy()
    rec = df.recognized.to_numpy()
    r, u = hit[rec], hit[~rec]
    rng = np.random.default_rng(seed)
    boots = [rng.choice(r, len(r)).mean() - rng.choice(u, len(u)).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"n_recognized": len(r), "acc_recognized": r.mean(), "acc_not": u.mean(),
            "gap": r.mean() - u.mean(), "ci_low": lo, "ci_high": hi}


gaps = pd.DataFrame({name: recog_gap(df) for name, df in
                     [("test 2024-26", b[b.split == "test"]), ("all years", b)]}).T
print(gaps.round(3).to_string())

g = gaps.loc["test 2024-26"]
enough = g.n_recognized >= A1_MIN_RECOGNIZED
t1 = enough and g.gap > A1_GAP and g.ci_low > 0
print(f"\nT1 fires: {t1}  (gap {g.gap:+.3f} vs > {A1_GAP}; CI lower bound {g.ci_low:+.3f} vs > 0; "
      f"n recognized {int(g.n_recognized)} vs >= {A1_MIN_RECOGNIZED})")
record("A-1", "T1 accuracy gap (test)", f"{g.gap:+.3f} (CI {g.ci_low:+.3f} to {g.ci_high:+.3f})",
       f"> {A1_GAP} and CI low > 0", t1, f"n recognized = {int(g.n_recognized)}")

              n_recognized  acc_recognized  acc_not    gap  ci_low  ci_high
test 2024-26          40.0           0.600    0.536  0.064   -0.10    0.209
all years            322.0           0.543    0.527  0.017   -0.04    0.074

T1 fires: False  (gap +0.064 vs > 0.05; CI lower bound -0.100 vs > 0; n recognized 40 vs >= 30)


### A-1 T2: sensitivity to recognized and leaked items

**Kappas:** recomputed on the audit sample with the 6 leaked items removed, the recognized items removed, and both removed.

**Model E:** B (objective + grade) and E (B + Claude fields) are fit as in `final_comparison.py`, with C tuned on 2023. Each is scored on the full test split and again with recognized or leaked players removed. The trigger compares Model E's macro-F1 before and after.

In [11]:
subsets = {"no leaked": ~ev.leaked, "no recognized": ~ev.recognized, "no leaked or recognized": ~(ev.leaked | ev.recognized)}
print(f"Audit items removed: leaked {int(ev.leaked.sum())}, recognized {int(ev.recognized.sum())}, "
      f"either {int((ev.leaked | ev.recognized).sum())}\n")

rows = []
for name, keep in subsets.items():
    d = (kappa_table(ev[keep]) - K_ALL)
    for f in FIELDS:
        for pair in PAIRS:
            rows.append({"subset": name, "field": f, "pair": pair, "kappa_all": K_ALL.loc[f, pair],
                         "kappa_subset": K_ALL.loc[f, pair] + d.loc[f, pair], "change": d.loc[f, pair]})
kchg = pd.DataFrame(rows)
# character_flag is excluded from the trigger: Claude gives it 0 positives, so its kappa is fixed at 0 (already dropped by H-1)
kchg_used = kchg[kchg.field != "character_flag"]
print("Largest kappa changes (character_flag excluded):")
print(kchg_used.reindex(kchg_used.change.abs().sort_values(ascending=False).index).head(8).round(3).to_string(index=False))
max_k = kchg_used.change.abs().max()

Audit items removed: leaked 6, recognized 5, either 8

Largest kappa changes (character_flag excluded):
                 subset          field         pair  kappa_all  kappa_subset  change
no leaked or recognized projected_tier Claude vs P1      0.735         0.702  -0.033
              no leaked projected_tier Claude vs P1      0.735         0.713  -0.023
          no recognized projected_tier Claude vs P1      0.735         0.715  -0.020
no leaked or recognized    injury_flag Claude vs P1      0.878         0.863  -0.015
no leaked or recognized    injury_flag     P1 vs P2      0.878         0.863  -0.015
              no leaked    injury_flag     P1 vs P2      0.878         0.864  -0.014
              no leaked    injury_flag Claude vs P1      0.878         0.864  -0.014
          no recognized    injury_flag     P1 vs P2      0.878         0.864  -0.014


In [22]:
a = FC.load().merge(b[C.KEY + ["recognized"]], on=C.KEY, how="left", validate="one_to_one")
leak_keys = ev.loc[ev.leaked, C.KEY].assign(leaked=True)
a = a.merge(leak_keys, on=C.KEY, how="left")
a["leaked"] = a.leaked.fillna(False).astype(bool)
tr, va, te = a[a.split == "train"], a[a.split == "val"], a[a.split == "test"]

_, pred_b = FC.tune_fit(M.OBJ_NUM + ["grade_pct"], tr, va, te)
_, pred_e = FC.tune_fit(M.OBJ_NUM + ["grade_pct"] + FC.CLAUDE_COLS, tr, va, te)
y = te.tier_code.to_numpy()

rows = []
for name, keep in {"full test": np.ones(len(te), bool), "no recognized": ~te.recognized.to_numpy(),
                   "no leaked": ~te.leaked.to_numpy(),
                   "no leaked or recognized": ~(te.recognized | te.leaked).to_numpy()}.items():
    f1b, f1e = E.macro_f1(y[keep], pred_b[keep]), E.macro_f1(y[keep], pred_e[keep])
    rows.append({"test rows": name, "n": int(keep.sum()), "macro_f1_B": f1b, "macro_f1_E": f1e, "E_minus_B": f1e - f1b})
mf = pd.DataFrame(rows)
mf["change_in_E"] = mf.macro_f1_E - mf.macro_f1_E.iloc[0]
mf["change_in_E_minus_B"] = mf.E_minus_B - mf.E_minus_B.iloc[0]
print(mf.round(3).to_string(index=False))
max_f1 = mf.change_in_E.abs().max()

t2 = max(max_k, max_f1) > A1_MDD
print(f"\nT2 fires: {t2}  (largest kappa change {max_k:.3f}; largest Model E macro-F1 change {max_f1:.3f}; MDD {A1_MDD})")
record("A-1", "T2 sensitivity (kappa / Model E F1)", f"kappa {max_k:.3f}; F1 {max_f1:.3f}", f"> {A1_MDD}", t2)

              test rows    n  macro_f1_B  macro_f1_E  E_minus_B  change_in_E  change_in_E_minus_B
              full test 1302       0.662       0.663      0.001        0.000                0.000
          no recognized 1262       0.657       0.655     -0.002       -0.008               -0.003
              no leaked 1296       0.661       0.662      0.001       -0.001               -0.000
no leaked or recognized 1259       0.658       0.657     -0.002       -0.007               -0.003

T2 fires: True  (largest kappa change 0.033; largest Model E macro-F1 change 0.008; MDD 0.029)


### A-1 T3: recognition rate by class year

In [13]:
by_year = b.groupby("year").agg(n=("recognized", "size"), recognized=("recognized", "sum"),
                                recognition_rate=("recognized", "mean"))
by_year["split"] = b.groupby("year").split.first()
by_year["over_threshold"] = by_year.recognition_rate > A1_RECOG_RATE
print(by_year.round(3).to_string())
fired_years = by_year.index[by_year.over_threshold].tolist()
t3 = bool(fired_years)
print(f"\nT3 fires: {t3}  years over {A1_RECOG_RATE:.0%}: {fired_years}")
record("A-1", "T3 recognition rate in any class year", f"max {by_year.recognition_rate.max():.3f} ({by_year.recognition_rate.idxmax()})",
       f"> {A1_RECOG_RATE}", t3, f"years over: {fired_years}")

        n  recognized  recognition_rate             split  over_threshold
year                                                                     
2014  439          32             0.073  holdout_nawrocki           False
2015  360          27             0.075             train           False
2016  363          41             0.113             train            True
2017  380          33             0.087             train           False
2018  464          24             0.052             train           False
2019  486          19             0.039             train           False
2020  491          34             0.069             train           False
2021  454          36             0.079             train           False
2022  455          20             0.044             train           False
2023  473          16             0.034               val           False
2024  459          16             0.035              test           False
2025  433          13             0.03

### A-1 item-level back-test: the rows the action removes

The A-1 action removes recognized and leak-flagged rows. How many rows is that, and are they where the errors are?

In [14]:
a1_flag_batch = b.recognized
show_backtest("Batch, flag = recognized (error = outcome miss)", a1_flag_batch, b.outcome_miss)
print()
_ = show_backtest("Audit sample, flag = recognized or leaked (error = Claude differs from both partners)",
                  ev.recognized | ev.leaked, ev.audit_error)

Batch, flag = recognized (error = outcome miss)
  Flags:   322/5667 items (5.7%)
  Catches: 147/2677 errors (5%)   precision of flag: 0.46

Audit sample, flag = recognized or leaked (error = Claude differs from both partners)
  Flags:   8/100 items (8.0%)
  Catches: 1/29 errors (3%)   precision of flag: 0.12


## Rule H-1: Human field review (human review)

On a random audit of **max(100 reports, 2% of the run)**, spread evenly across real draft tiers and labeled by at least two people, a field fires if:
- **T1:** the inter-reviewer κ (P1 vs P2) is **< 0.60**, or
- **T2:** Claude's κ against **either** partner is **< 0.60**.

**Action:** review the rubric for that field and remove it from the analysis until a later audit sample meets the threshold.

κ is quadratic-weighted for the ordered fields (tier, role, tone) and unweighted Cohen's κ for the 0/1 flags. The existing eval set (25 items per real tier, 2024–26) serves as the audit sample.

In [15]:
need = max(H1_AUDIT_MIN, int(np.ceil(H1_AUDIT_SHARE * len(b))))
print(f"Audit size required: max({H1_AUDIT_MIN}, {H1_AUDIT_SHARE:.0%} x {len(b):,}) = {need};  audit used: {len(ev)}"
      f"  -> {'meets' if len(ev) >= need else 'BELOW'} the rule's sample size")
print(f"Per real tier: {ev.tier.value_counts().reindex(TIERS).to_dict()}\n")

h1 = K_ALL.copy()
h1["min_Claude"] = h1[["Claude vs P1", "Claude vs P2"]].min(axis=1)
h1["T1_human_fires"] = h1["P1 vs P2"] < H1_KAPPA
h1["T2_llm_fires"] = h1["min_Claude"] < H1_KAPPA
# what share of audit items Claude gets "wrong" on each field (differs from both partners)
h1["claude_differs_from_both"] = [((field_codes(ev, f, "cl") != field_codes(ev, f, "1")) &
                                   (field_codes(ev, f, "cl") != field_codes(ev, f, "2"))).mean() for f in FIELDS]
print(h1.round(3).to_string())

dropped = h1.index[h1.T1_human_fires | h1.T2_llm_fires].tolist()
print(f"\nFields flagged: {len(dropped)}/{len(FIELDS)} -> {dropped}")
record("H-1", "T1 human-human kappa < 0.60", f"min {h1['P1 vs P2'].min():.3f}", f"< {H1_KAPPA}", h1.T1_human_fires.any(),
       f"fields: {h1.index[h1.T1_human_fires].tolist()}")
record("H-1", "T2 Claude-human kappa < 0.60", f"min {h1.min_Claude.min():.3f}", f"< {H1_KAPPA}", h1.T2_llm_fires.any(),
       f"fields: {h1.index[h1.T2_llm_fires].tolist()}; audit n = {len(ev)} vs required {need}")

Audit size required: max(100, 2% x 5,667) = 114;  audit used: 100  -> BELOW the rule's sample size
Per real tier: {'R1': 25, 'R2-3': 25, 'R4-7': 25, 'UDFA': 25}

                P1 vs P2  Claude vs P1  Claude vs P2  min_Claude  T1_human_fires  T2_llm_fires  claude_differs_from_both
projected_tier     0.907         0.735         0.795       0.735           False         False                      0.29
role               0.883         0.758         0.834       0.758           False         False                      0.21
tone               0.819         0.711         0.711       0.711           False         False                      0.33
injury_flag        0.878         0.878         1.000       0.878           False         False                      0.00
character_flag     0.918         0.000         0.000       0.000           False          True                      0.06

Fields flagged: 1/5 -> ['character_flag']


## Rule H-2: Human output review (human review)

Each row is flagged if it has **any** of:
1. an **error output**: the response didn't finish normally (`stop_reason` ≠ `end_turn`) or a required field is missing;
2. an **`evidence` quote that isn't word for word** in the masked report (whitespace, case and curly quotes are normalized first);
3. a **Claude tier 2 or more tiers away from the grade-implied tier**.

The rule fires when **more than 5%** of a run's rows are flagged.

In [16]:
REQUIRED = ["projected_tier", "role", "injury_flag", "character_flag", "tone", "evidence", "recognized"]


def norm(s):
    s = str(s).replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return " ".join(s.split()).lower().strip(' ."')


def h2_flags(df, tier_gap=H2_TIER_GAP):
    f = pd.DataFrame(index=df.index)
    f["error_output"] = (df.stop_reason != "end_turn") | df[REQUIRED].isna().any(axis=1)
    f["not_verbatim"] = [norm(e) not in norm(t) for e, t in zip(df.evidence, df.text_masked)]
    f["grade_conflict"] = (df.cl - df.grade_t).abs() >= tier_gap
    f["any"] = f.any(axis=1)
    return f


fb, fe = h2_flags(b), h2_flags(ev)
print("Batch: rows flagged by each condition")
print(pd.DataFrame({"rows": fb.sum(), "share": fb.mean().round(4)}).to_string())
rate = fb["any"].mean()
fires = rate > H2_ROW_RATE
print(f"\nH-2 fires: {fires}  (flagged share {rate:.1%} vs > {H2_ROW_RATE:.0%})\n")
record("H-2", "share of rows flagged", f"{rate:.3f} ({int(fb['any'].sum())} rows)", f"> {H2_ROW_RATE}", fires)

show_backtest("Batch (error = outcome miss)", fb["any"], b.outcome_miss)
print()
show_backtest("Audit sample (error = Claude differs from both partners)", fe["any"], ev.audit_error)
print("\nPer condition, audit sample:")
for c in ["error_output", "not_verbatim", "grade_conflict"]:
    print(f"  {c:15s} flags {int(fe[c].sum()):3d}   catches {int((fe[c] & ev.audit_error).sum())}/{int(ev.audit_error.sum())}")

Batch: rows flagged by each condition
                rows   share
error_output       0  0.0000
not_verbatim      96  0.0169
grade_conflict    31  0.0055
any              127  0.0224

H-2 fires: False  (flagged share 2.2% vs > 5%)

Batch (error = outcome miss)
  Flags:   127/5667 items (2.2%)
  Catches: 79/2677 errors (3%)   precision of flag: 0.62

Audit sample (error = Claude differs from both partners)
  Flags:   3/100 items (3.0%)
  Catches: 0/29 errors (0%)   precision of flag: 0.00

Per condition, audit sample:
  error_output    flags   0   catches 0/29
  not_verbatim    flags   1   catches 0/29
  grade_conflict  flags   2   catches 0/29


In [17]:
# A sample of the non-verbatim quotes, for the "confirm misquotes are reasonable" step of the action
b.loc[fb.not_verbatim, C.KEY[1:] + ["projected_tier", "evidence"]].head(8)

,year,position,projected_tier,evidence
84,2016,DT,R1,his ceiling appears to be high with a chance t...
165,2018,EDGE,R4-7,teams may view him as a designated pass rusher...
220,2016,DE,R1,should make him a very good 4-3 base end or a ...
223,2015,CB,R2-3,"Has the desired size, length and playmaking qu..."
297,2024,DT,R2-3,his size and skill set will have him plugged i...
407,2024,G,R4-7,his size and length could prevent it... future...
517,2023,LB,UDFA,physical but limited prospect... he's missing ...
563,2016,ILB,R4-7,looks the part of a backup WILL inside linebac...


## Rule B-1: School-status disparity (bias & fairness)

Within each **real** draft tier, compare Power 5 and non-Power 5 players on three rates:
- **error:** Claude's tier differs from the real tier;
- **under:** Claude's tier is lower than the real tier (not defined for UDFA);
- **over:** Claude's tier is higher than the real tier (not defined for R1).

**Disparity ratio** = higher rate ÷ lower rate (the group with the lower rate goes in the denominator). The same ratio is computed with the **grade-implied tier** in place of Claude's tier, **keeping the same two groups in the same positions** so the two ratios point the same way.

- **T1:** Claude's ratio is **> 1.25** → disclose.
- **T2:** Claude's ratio exceeds the grade's ratio by **0.10 or more** → pause the workflow (when T1 also fires).

Each tier uses the 2024–26 test split when both groups have at least 30 players, and all analysis years otherwise (the `scope` column).

In [18]:
def rates(df, pred):
    return {"error": (df[pred] != df.real).mean(),
            "under": (df[pred] > df.real).mean(),   # higher code = later tier
            "over": (df[pred] < df.real).mean()}


rows = []
for t, tier in enumerate(TIERS):
    test_cell = b[(b.split == "test") & (b.real == t)]
    use_test = test_cell.groupby("power5").size().reindex([False, True], fill_value=0).min() >= B1_MIN_N
    cell = test_cell if use_test else b[b.real == t]
    grp = {k: cell[cell.power5 == k] for k in (False, True)}
    cl_r = {k: rates(g_, "cl") for k, g_ in grp.items()}
    gr_r = {k: rates(g_, "grade_t") for k, g_ in grp.items()}
    for m in ["error", "under", "over"]:
        if (m == "under" and tier == "UDFA") or (m == "over" and tier == "R1"):
            continue
        lo_grp = min(grp, key=lambda k: cl_r[k][m])          # denominator = group with lower Claude rate
        hi_grp = not lo_grp
        ratio = lambda r: r[hi_grp][m] / r[lo_grp][m] if r[lo_grp][m] > 0 else np.inf
        rows.append({"tier": tier, "rate": m, "scope": "test" if use_test else "all years",
                     "n_nonP5": len(grp[False]), "n_P5": len(grp[True]),
                     "higher_group": "P5" if hi_grp else "non-P5",
                     # cases behind the denominator: a ratio resting on 1-2 cases is unstable
                     "lower_group_cases": int(round(cl_r[lo_grp][m] * len(grp[lo_grp]))),
                     "claude_nonP5": cl_r[False][m], "claude_P5": cl_r[True][m], "claude_ratio": ratio(cl_r),
                     "grade_ratio": ratio(gr_r)})
b1 = pd.DataFrame(rows)
b1["excess"] = b1.claude_ratio - b1.grade_ratio
b1["T1_fires"] = b1.claude_ratio > B1_RATIO
b1["T2_fires"] = b1.excess >= B1_EXCESS
print(b1.round(3).to_string(index=False))

n1, n2, both = b1.T1_fires.sum(), b1.T2_fires.sum(), (b1.T1_fires & b1.T2_fires).sum()
print(f"\nCells checked: {len(b1)}   T1 fires in {n1}   T2 fires in {n2}   both in {both}")
print("Action:", "PAUSE workflow (T1 and T2 both fire)" if both else ("DISCLOSE (T1 only)" if n1 else "none"))
record("B-1", "T1 disparity ratio > 1.25", f"{n1}/{len(b1)} cells; max {b1.claude_ratio.max():.2f}", f"> {B1_RATIO}", n1 > 0)
record("B-1", "T2 Claude ratio - grade ratio >= 0.10", f"{n2}/{len(b1)} cells; max {b1.excess.max():+.2f}",
       f">= {B1_EXCESS}", n2 > 0, f"cells where both fire: {both}")

tier  rate     scope  n_nonP5  n_P5 higher_group  lower_group_cases  claude_nonP5  claude_P5  claude_ratio  grade_ratio  excess  T1_fires  T2_fires
  R1 error all years       31   383       non-P5                228         0.742      0.595         1.246        1.911  -0.664     False     False
  R1 under all years       31   383       non-P5                228         0.742      0.595         1.246        1.911  -0.664     False     False
R2-3 error all years      158   746       non-P5                387         0.639      0.519         1.232        0.974   0.258     False      True
R2-3 under all years      158   746       non-P5                331         0.601      0.444         1.355        1.301   0.054      True     False
R2-3  over all years      158   746           P5                  6         0.038      0.075         1.977        2.800  -0.824      True     False
R4-7 error      test       62   396           P5                 15         0.242      0.293         1.211      

## Threshold sensitivity

Try 2–3 values for each threshold. If a small change flips the result, the threshold is fragile, and the memo should say so.

In [19]:
print("A-1 T1 gap threshold (test gap = %+.3f, CI low = %+.3f):" % (g.gap, g.ci_low))
for th in [0.03, 0.05, 0.10]:
    print(f"  gap > {th:.2f} and CI low > 0: {bool(enough and g.gap > th and g.ci_low > 0)}")

print("A-1 T3 recognition-rate threshold:")
for th in [0.05, 0.10, 0.15]:
    print(f"  > {th:.0%}: years {by_year.index[by_year.recognition_rate > th].tolist()}")

print("H-1 kappa floor:")
for th in [0.50, 0.60, 0.70]:
    print(f"  < {th:.2f}: fields {h1.index[(h1['P1 vs P2'] < th) | (h1.min_Claude < th)].tolist()}")

print("H-2 tier-gap condition (row flag rate on batch; audit errors caught):")
for gap in [1, 2, 3]:
    fb_, fe_ = h2_flags(b, gap)["any"], h2_flags(ev, gap)["any"]
    print(f"  gap >= {gap}: flags {fb_.mean():5.1%} of batch;  audit catches {int((fe_ & ev.audit_error).sum())}/{int(ev.audit_error.sum())}"
          f"  -> rule {'fires' if fb_.mean() > H2_ROW_RATE else 'quiet'} at {H2_ROW_RATE:.0%}")

print("B-1 thresholds (cells firing out of %d):" % len(b1))
for th in [1.10, 1.25, 1.50]:
    print(f"  ratio > {th:.2f}: {int((b1.claude_ratio > th).sum())}")
for th in [0.05, 0.10, 0.20]:
    print(f"  excess >= {th:.2f}: {int((b1.excess >= th).sum())}")

A-1 T1 gap threshold (test gap = +0.064, CI low = -0.100):
  gap > 0.03 and CI low > 0: False
  gap > 0.05 and CI low > 0: False
  gap > 0.10 and CI low > 0: False
A-1 T3 recognition-rate threshold:
  > 5%: years [2014, 2015, 2016, 2017, 2018, 2020, 2021]
  > 10%: years [2016]
  > 15%: years []
H-1 kappa floor:
  < 0.50: fields ['character_flag']
  < 0.60: fields ['character_flag']
  < 0.70: fields ['character_flag']
H-2 tier-gap condition (row flag rate on batch; audit errors caught):
  gap >= 1: flags 37.1% of batch;  audit catches 19/29  -> rule fires at 5%
  gap >= 2: flags  2.2% of batch;  audit catches 0/29  -> rule quiet at 5%
  gap >= 3: flags  1.7% of batch;  audit catches 0/29  -> rule quiet at 5%
B-1 thresholds (cells firing out of 10):
  ratio > 1.10: 10
  ratio > 1.25: 6
  ratio > 1.50: 2
  excess >= 0.05: 4
  excess >= 0.10: 3
  excess >= 0.20: 2


## Summary: one back-test line per rule

Copy these into the memo, one line under each rule.

In [20]:
summary = pd.DataFrame(SUMMARY)
summary

,rule,trigger,value,threshold,fired,note
0,A-1,T1 accuracy gap (test),+0.064 (CI -0.100 to +0.209),> 0.05 and CI low > 0,False,n recognized = 40
1,A-1,T2 sensitivity (kappa / Model E F1),kappa 0.033; F1 0.008,> 0.029,True,
2,A-1,T3 recognition rate in any class year,max 0.113 (2016),> 0.1,True,years over: [2016]
3,H-1,T1 human-human kappa < 0.60,min 0.819,< 0.6,False,fields: []
4,H-1,T2 Claude-human kappa < 0.60,min 0.000,< 0.6,True,fields: ['character_flag']; audit n = 100 vs r...
5,H-2,share of rows flagged,0.022 (127 rows),> 0.05,False,
6,B-1,T1 disparity ratio > 1.25,6/10 cells; max 8.45,> 1.25,True,
7,B-1,T2 Claude ratio - grade ratio >= 0.10,3/10 cells; max +2.74,>= 0.1,True,cells where both fire: 1


## Wrap-up

1. **Change a threshold** in the Setup cell and re-run to see how the result moves.
2. **Add a back-test line under each rule in the memo:** flag rate, errors caught, and anything you changed.
3. **Update your AI use log** if an AI-drafted threshold didn't hold up against the data.

In [21]:
print("#" * 60)
print("Checklist:")
print("  [ ] Back-tested every memo threshold on my own project data")
print("  [ ] Checked flag rate AND how many errors each rule catches")
print("  [ ] Tried 2-3 values for each threshold")
print("  [ ] Back-test cells copied into my final project notebook")
print("  [ ] Back-test line added under each rule in my memo")
print("  [ ] AI use log updated if a threshold changed")
print("#" * 60)

############################################################
Checklist:
  [ ] Back-tested every memo threshold on my own project data
  [ ] Checked flag rate AND how many errors each rule catches
  [ ] Tried 2-3 values for each threshold
  [ ] Back-test cells copied into my final project notebook
  [ ] Back-test line added under each rule in my memo
  [ ] AI use log updated if a threshold changed
############################################################


# Appendix: `governance_checks.py` replicated on Claude's labels

`src/governance_checks.py` (notebook sections 10–12) judges **Model D's** errors on the 2024–26 test split. The cells below run the same three sections, but the prediction being judged is **Claude's `projected_tier`** (an error = Claude's tier differs from the real draft tier):

- **Section 10, subgroups:** error rate by position group, Power 5 status, class year and grade band (thirds of `grade_pct` on the test split), with the disparate-impact ratio (flag > 1.25), a chi-square test, and the equalized-odds recall spread by position group (unequal > 0.20). Model D's error rate is shown beside each group for comparison.
- **Section 11, cost:** unchanged. It already describes the Claude runs.
- **Section 12, review-rule back-tests:** the same four rules, scored against Claude's errors (and against Model D's errors for reference).

To judge Model E instead, set `PRED = "pred_E"` in the next cell.

In [23]:
from src import governance_checks as GC

PRED = "cl_tier_code"        # Claude's projected tier; use "pred_E" for Model E, "pred_D" to reproduce governance_checks.py
PRED_NAME = {"cl_tier_code": "Claude tier", "pred_E": "Model E", "pred_D": "Model D"}[PRED]

t = GC.test_frame()          # test split with pred_D, pred_E, Claude fields and 'recognized' (fits D and E; takes a minute)
t["err_pred"] = t[PRED].to_numpy() != t.tier_code.to_numpy()
t["err_D"] = t.pred_D.to_numpy() != t.tier_code.to_numpy()
edges = pd.qcut(t.grade_pct, 3, retbins=True)[1]
print(f"Test split n = {len(t)}.  {PRED_NAME} errors = {int(t.err_pred.sum())} ({t.err_pred.mean():.3f});  "
      f"Model D errors = {int(t.err_D.sum())} ({t.err_D.mean():.3f})")
print(f"Grade-band cut points (test-split tertiles of grade_pct): {np.round(edges, 3).tolist()}")

Test split n = 1302.  Claude tier errors = 601 (0.462);  Model D errors = 452 (0.347)
Grade-band cut points (test-split tertiles of grade_pct): [0.004, 0.336, 0.68, 1.0]


## 10. Subgroup error rates

In [24]:
SUBGROUPS = ["positionGroup", "power5", "year_str", "grade_band"]
sub_rows = []
for col in SUBGROUPS:
    g = t[col].astype(str)
    tab = E.subgroup_errors(t.err_pred, g)
    tab["model_D_error_rate"] = t.err_D.groupby(g, observed=True).mean().reindex(tab.index)
    chi = E.chi_square_errors(t.err_pred, g)
    print(f"### {col}   (chi-square p = {chi['p']:.3f}, dof {chi['dof']}, min expected {chi['min_expected']:.1f})")
    print(tab.round(3).to_string(), "\n")
    sub_rows.append({"subgroup": col, "max_di_ratio": tab.di_ratio.max(), "groups_flagged": int(tab["flag_1.25"].sum()),
                     "chi2_p": chi["p"], "significant_at_0.05": chi["p"] < 0.05})

odds = E.equalized_odds_recall(pd.Series(t.tier_code.to_numpy()), pd.Series(t[PRED].to_numpy()),
                               pd.Series(t.positionGroup.astype(str).to_numpy()))
odds["tier"] = [TIERS[int(c)] for c in odds["class"]]
print(f"### equalized odds by positionGroup ({PRED_NAME}; recall spread > 0.20 = unequal)")
print(odds.round(3).to_string(index=False))

### positionGroup   (chi-square p = 0.768, dof 7, min expected 24.9)
                 n  errors  error_rate  di_ratio  flag_1.25  model_D_error_rate
positionGroup                                                                  
WR             182      78       0.429     1.000      False               0.357
DL             268     117       0.437     1.019      False               0.392
RB             105      46       0.438     1.022      False               0.314
DB             258     118       0.457     1.067      False               0.357
LB             111      53       0.477     1.114      False               0.333
TE              85      42       0.494     1.153      False               0.341
QB              54      27       0.500     1.167      False               0.315
OL             239     120       0.502     1.172      False               0.310 

### power5   (chi-square p = 0.008, dof 1, min expected 113.1)
           n  errors  error_rate  di_ratio  flag_1.25  model_D_err

In [25]:
sub_summary = pd.DataFrame(sub_rows)
sub_summary.round(3)

,subgroup,max_di_ratio,groups_flagged,chi2_p,significant_at_0.05
0,positionGroup,1.172,0,0.768,False
1,power5,1.250,1,0.008,True
2,year_str,1.102,0,0.405,False
3,grade_band,1.310,2,0.000,True


## 11. Cost

In [26]:
GC.cost_table()

,run,API,$
0,"pilot v1 (2023, 52 items)",standard,0.22000
1,"pilot v2 (2023, 52 items)",standard,0.22000
2,"eval v2 (2024-26, 100 items)",standard,0.43000
3,"batch v2 (analysis set, 5,667)",batch,11.62000
4,total spent,,12.49000
5,batch set at standard prices (not paid),standard,23.24000
6,"per report, batch",batch,0.00205


## 12. Review-rule back-tests

The four rules from `governance_checks.py`, unchanged. Each is scored twice: against the errors of the prediction chosen above, and against Model D's errors (which reproduces `docs/subgroups_costs_governance.md` exactly).

In [27]:
bt_rules = {
    "1: abs(Claude tier - model tier) >= 2": np.abs(t.cl_tier_code.to_numpy() - t.pred_D.to_numpy()) >= 2,
    "2: Claude tier != model tier": t.cl_tier_code.to_numpy() != t.pred_D.to_numpy(),
    "3: Claude recognized = true": t.recognized.fillna(False).astype(bool).to_numpy(),
    "4: Claude injury_flag = 1": t.cl_injury.to_numpy() == 1,
}
rows = []
for name, flag in bt_rules.items():
    for label, err in [(PRED_NAME, t.err_pred), ("Model D", t.err_D)]:
        r = E.backtest_flag(pd.Series(flag), pd.Series(err.to_numpy()))
        rows.append({"rule": name, "errors of": label, **r})
bt = pd.DataFrame(rows)
bt.round(3)

,rule,errors of,flag_rate,flagged,errors,caught,catch_rate,precision_of_flag
0,1: abs(Claude tier - model tier) >= 2,Claude tier,0.008,11,601,11,0.018,1.000
1,1: abs(Claude tier - model tier) >= 2,Model D,0.008,11,452,5,0.011,0.455
2,2: Claude tier != model tier,Claude tier,0.433,564,601,374,0.622,0.663
3,2: Claude tier != model tier,Model D,0.433,564,452,225,0.498,0.399
4,3: Claude recognized = true,Claude tier,0.031,40,601,16,0.027,0.400
5,3: Claude recognized = true,Model D,0.031,40,452,11,0.024,0.275
6,4: Claude injury_flag = 1,Claude tier,0.095,124,601,66,0.110,0.532
7,4: Claude injury_flag = 1,Model D,0.095,124,452,49,0.108,0.395


In [28]:
for name in bt_rules:
    r = bt[(bt.rule == name) & (bt["errors of"] == PRED_NAME)].iloc[0]
    print(f"Rule {name}\n  Back-test: on {len(t):,} test reports, flags {r.flag_rate:.1%} ({r.flagged}) and catches "
          f"{r.caught} of {r.errors} {PRED_NAME} errors ({r.catch_rate:.0%}); precision {r.precision_of_flag:.2f}")

Rule 1: abs(Claude tier - model tier) >= 2
  Back-test: on 1,302 test reports, flags 0.8% (11) and catches 11 of 601 Claude tier errors (2%); precision 1.00
Rule 2: Claude tier != model tier
  Back-test: on 1,302 test reports, flags 43.3% (564) and catches 374 of 601 Claude tier errors (62%); precision 0.66
Rule 3: Claude recognized = true
  Back-test: on 1,302 test reports, flags 3.1% (40) and catches 16 of 601 Claude tier errors (3%); precision 0.40
Rule 4: Claude injury_flag = 1
  Back-test: on 1,302 test reports, flags 9.5% (124) and catches 66 of 601 Claude tier errors (11%); precision 0.53


## Appendix, all years (2014–2026)

The same subgroup checks and review rules, run on all 5,667 analysis reports instead of only the 2024–26 test split. Uses `b` from the top of the notebook.

**What changes from the test-split version:**
- **The comparison is the grade-implied tier, not Model D.** Model D is fit on 2015–22, so its predictions for those years are in-sample, and comparing them with Claude would favor the model. The grade-implied tier (analyst grade percentile cut at the 2015–22 tier shares, defined at the top) has no fitted parameters other than the cut points, so it works for every year.
- **Rules 1 and 2 compare Claude's tier with the grade-implied tier** (marked `*`), not with Model D's tier. Rules 3 and 4 are unchanged.
- **Grade bands** are thirds of `grade_pct` across all years.
- **Section 11 (cost)** is the same for every scope, so it isn't repeated.
- **Contamination:** in the training years Claude may have seen more coverage of these players' drafts (A-1 T3 found 2016 above 10% recognition), so treat the earlier years with that in mind.

In [29]:
ta = b.copy()
ta["err_pred"] = ta.cl != ta.real          # Claude tier vs real tier
ta["err_ref"] = ta.grade_t != ta.real      # grade-implied tier vs real tier
ta["year_str"] = ta.year.astype(str)
ta["grade_band"] = pd.qcut(ta.grade_pct, 3, labels=["low", "mid", "high"])
edges_all = pd.qcut(ta.grade_pct, 3, retbins=True)[1]
REF_NAME = "grade-implied tier"
print(f"All analysis reports n = {len(ta):,}.  Claude tier errors = {int(ta.err_pred.sum()):,} ({ta.err_pred.mean():.3f});  "
      f"{REF_NAME} errors = {int(ta.err_ref.sum()):,} ({ta.err_ref.mean():.3f})")
print(f"Grade-band cut points: {np.round(edges_all, 3).tolist()}")
print("Rows by split:", ta.split.value_counts().to_dict())

All analysis reports n = 5,667.  Claude tier errors = 2,677 (0.472);  grade-implied tier errors = 2,007 (0.354)
Grade-band cut points: [0.002, 0.336, 0.656, 1.0]
Rows by split: {'train': 3453, 'test': 1302, 'val': 473, 'holdout_nawrocki': 439}


### 10. Subgroup error rates, all years

In [30]:
sub_rows_all = []
for col in ["positionGroup", "power5", "year_str", "grade_band"]:
    g = ta[col].astype(str)
    tab = E.subgroup_errors(ta.err_pred, g)
    tab[f"{REF_NAME} error_rate"] = ta.err_ref.groupby(g, observed=True).mean().reindex(tab.index)
    chi = E.chi_square_errors(ta.err_pred, g)
    print(f"### {col}   (chi-square p = {chi['p']:.3f}, dof {chi['dof']}, min expected {chi['min_expected']:.1f})")
    print(tab.round(3).to_string(), "\n")
    sub_rows_all.append({"subgroup": col, "max_di_ratio": tab.di_ratio.max(), "groups_flagged": int(tab["flag_1.25"].sum()),
                         "chi2_p": chi["p"], "significant_at_0.05": chi["p"] < 0.05})

odds_all = E.equalized_odds_recall(pd.Series(ta.real.to_numpy()), pd.Series(ta.cl.to_numpy()),
                                   pd.Series(ta.positionGroup.astype(str).to_numpy()))
odds_all["tier"] = [TIERS[int(c)] for c in odds_all["class"]]
print("### equalized odds by positionGroup (Claude tier; recall spread > 0.20 = unequal)")
print(odds_all.round(3).to_string(index=False))

### positionGroup   (chi-square p = 0.611, dof 7, min expected 119.0)
                  n  errors  error_rate  di_ratio  flag_1.25  grade-implied tier error_rate
positionGroup                                                                              
LB              576     253       0.439     1.000      False                          0.361
RB              529     241       0.456     1.037      False                          0.335
QB              252     116       0.460     1.048      False                          0.325
OL              983     463       0.471     1.072      False                          0.355
DL             1065     502       0.471     1.073      False                          0.356
DB             1105     535       0.484     1.102      False                          0.381
TE              350     170       0.486     1.106      False                          0.323
WR              807     397       0.492     1.120      False                          0.344 

### powe

In [31]:
pd.DataFrame(sub_rows_all).round(3)

,subgroup,max_di_ratio,groups_flagged,chi2_p,significant_at_0.05
0,positionGroup,1.120,0,0.611,False
1,power5,1.036,0,0.278,False
2,year_str,1.410,3,0.000,True
3,grade_band,1.398,2,0.000,True


### 12. Review-rule back-tests, all years

In [32]:
bt_rules_all = {
    "1*: abs(Claude tier - grade-implied tier) >= 2": np.abs(ta.cl.to_numpy() - ta.grade_t.to_numpy()) >= 2,
    "2*: Claude tier != grade-implied tier": ta.cl.to_numpy() != ta.grade_t.to_numpy(),
    "3: Claude recognized = true": ta.recognized.to_numpy(),
    "4: Claude injury_flag = 1": ta.injury_flag.to_numpy() == 1,
}
rows = []
for name, flag in bt_rules_all.items():
    for label, err in [("Claude tier", ta.err_pred), (REF_NAME, ta.err_ref)]:
        rows.append({"rule": name, "errors of": label, **E.backtest_flag(pd.Series(flag), pd.Series(err.to_numpy()))})
bt_all = pd.DataFrame(rows)
bt_all.round(3)

,rule,errors of,flag_rate,flagged,errors,caught,catch_rate,precision_of_flag
0,1*: abs(Claude tier - grade-implied tier) >= 2,Claude tier,0.005,31,2677,30,0.011,0.968
1,1*: abs(Claude tier - grade-implied tier) >= 2,grade-implied tier,0.005,31,2007,10,0.005,0.323
2,2*: Claude tier != grade-implied tier,Claude tier,0.359,2035,2677,1417,0.529,0.696
3,2*: Claude tier != grade-implied tier,grade-implied tier,0.359,2035,2007,747,0.372,0.367
4,3: Claude recognized = true,Claude tier,0.057,322,2677,147,0.055,0.457
5,3: Claude recognized = true,grade-implied tier,0.057,322,2007,116,0.058,0.360
6,4: Claude injury_flag = 1,Claude tier,0.140,792,2677,384,0.143,0.485
7,4: Claude injury_flag = 1,grade-implied tier,0.140,792,2007,288,0.143,0.364


In [33]:
for name in bt_rules_all:
    r = bt_all[(bt_all.rule == name) & (bt_all["errors of"] == "Claude tier")].iloc[0]
    print(f"Rule {name}\n  Back-test: on {len(ta):,} reports (2014-26), flags {r.flag_rate:.1%} ({r.flagged:,}) and catches "
          f"{r.caught:,} of {r.errors:,} Claude tier errors ({r.catch_rate:.0%}); precision {r.precision_of_flag:.2f}")

Rule 1*: abs(Claude tier - grade-implied tier) >= 2
  Back-test: on 5,667 reports (2014-26), flags 0.5% (31) and catches 30 of 2,677 Claude tier errors (1%); precision 0.97
Rule 2*: Claude tier != grade-implied tier
  Back-test: on 5,667 reports (2014-26), flags 35.9% (2,035) and catches 1,417 of 2,677 Claude tier errors (53%); precision 0.70
Rule 3: Claude recognized = true
  Back-test: on 5,667 reports (2014-26), flags 5.7% (322) and catches 147 of 2,677 Claude tier errors (5%); precision 0.46
Rule 4: Claude injury_flag = 1
  Back-test: on 5,667 reports (2014-26), flags 14.0% (792) and catches 384 of 2,677 Claude tier errors (14%); precision 0.48
